# Turbulence synthesis: MGD and MGD regularised (Guth method) 

This notebook runs the 1D turbulence synthesis experiment and compares MGD theta trajectories with the regularised trajectories. 


In [ ]:
import torch
import numpy as np
import matplotlib as mpl
%matplotlib inline
import matplotlib.pyplot as plt
from scipy import stats, ndimage
import math 

import sys
from pathlib import Path

import hashlib
from datetime import datetime
from types import SimpleNamespace

root = Path.cwd()
print(root)

sys.path.insert(0, str(root / '../codes'))
from sde_routines_condi import *
from sde_routines import *
from potentials_builder import *
from filters_bank import * 
from utils import *
from utils_entropy import *
from utils_experiment import * 
from check_moments import *
from potentials import *
from filters import *
from mala import *
from ortho_wavelet import *

sys.path.insert(0, str(root / '../data'))
from data_loader import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

root = Path.cwd()
print(root) 

## Data preprocessing

In [ ]:
# Orthogonal wavelet
W = DefineWavelet('Db', m=3, device=device)

Data = load_turbulence_1d()
print(Data.shape) 

n = 1024
Data = split_periodize_reshape(Data, n)

for j in range(3):
    Data = W.decompose(Data)[1]

n1 = 50
x1 = normalize(Data[:n1]).to(device) # choose here if you want less data 
print('Data shape after preprocessing:', Data.shape)
print('x1 shape:', x1.shape) 

In [ ]:
%matplotlib inline 
plot_time_series_row(x1, 10)

## Scaling exponent ratio of original data

In [ ]:
# scaling exponent ratio 
%matplotlib inline 
scaling_exponent_ratio_compare(x1, x1, 30, kill_points=1)

## Shared model/SDE parameters

In [ ]:
B, channels, M = x1.shape
print(f'(B, C, T) = ({B}, {channels}, {M}).')

J = 7
Q = 3

filters, filters_Phi = return_Filters(M, J, 1, device=device, include_phi=True)
filters_Q = return_Filters(M, J, Q, device=device)

print("filters shape:", filters.shape) 
print("filters_Q shape:", filters_Q.shape) 

interpolant = 'Cos'
nt = 119

# originally 50000
t = 1 - (1 - torch.linspace(0, 1, nt + 1))**2  # LINEAR SCHEDULE 
sigma = 3.5 

nb_workers = x1.shape[0]
nb_interpolants = x1.shape[0]
batch_size = x1.shape[0] 
regularization = 1e-2

n_subsample = 1
lam = 5e-07

seed = 100

terms = [
    "L_6",
    "L_6_psi", 
    "L_2_lowpass",
    "Scattering_Fourth_Order_Mod2_Real_Q1",
    "Scattering_Fourth_Order_Mod2_Imag_Q1",
    "Scalar_psi_gaussianK",
    "Scalar_morlet_gaussianK", 
]

timestamp= datetime.now().strftime("%Y%m%d_%H%M")

FORCE_RERUN = False
SAVE_AUX_MOMENTS = True

label = ""

args = SimpleNamespace(
       Re_number=None, timestamp=timestamp, label=label,
       J=J, Q=Q, terms=terms,
       nt=nt, sigma=sigma, interpolant=interpolant,
       regularization=regularization, lam=lam, n_subsample=n_subsample,
       batch_size=None, force_rerun=FORCE_RERUN, no_save_aux_moments=not SAVE_AUX_MOMENTS,
       n_traj_groups=5, n_tau=30, moment_threshold=1e-8, n1=x1.shape[0], seed=seed,
   )

config, exp_dir, fig_dir, potentials_dir, logger, loaded = resolve_or_setup_experiment_output(
    root, args, M, device,
    extra_metadata={'M': M, 'B': B, 'channels': channels},
    include_potentials_dir=True,
)

## Check wavelet coefficients histograms, Q=3

In [ ]:
%matplotlib inline 
wt = torch.fft.ifft(torch.fft.fft(x1) * filters_Q).real  # (B, J, T)
print(wt.shape) 
n_wavelets = filters_Q.shape[1]
# ------------------------------------------------------------------
# 1. Overview grid
# ------------------------------------------------------------------
ncols = 5
nrows = math.ceil(n_wavelets / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()
for i in range(n_wavelets):
    vals = wt[:, i, :].detach().cpu().flatten().numpy()
    axes[i].hist(vals, bins=50, density=True, log=True)
    axes[i].set_title(f"ch={i}")
    axes[i].set_xlabel("Coefficient value")
    axes[i].set_ylabel("Density")
for j in range(n_wavelets, len(axes)):
    axes[j].axis("off")
plt.suptitle(f"Wavelet coefficient histograms", fontsize=25)
plt.tight_layout()
plt.show()

## Check how the model fits data

In [ ]:
model1 = Scalar_GGD_KRegion.fit_and_compare(x1, M, J, 1, device)
model3 = Scalar_GGD_KRegion.fit_and_compare(x1, M, J, 3, device)

## Run or load an experiment
Set `FORCE_RERUN = True` if you want to recompute samples even when saved results already exist.

The standard outputs are saved with your existing `save_results(...)`. In addition, this notebook saves a small auxiliary `.pt` file containing `barphi_e` and `barphi_p`, because these are useful for plotting moment matching after reloading.

### find where $t_i$ switches to 1.0 and choose that as final time point for the run
Otherwise the regularized results will be $Nan$

In [ ]:
t_rounded = torch.round(t, decimals=4)
t_final = int((t_rounded == 1.0).nonzero(as_tuple=True)[0][0]) 
print(f"t_final = {t_final}/{len(t)} (last t = {t[t_final-1].item():.6f}, "
      f"dropping {len(t) - t_final} redundant trailing points at 1.0000)")

In [ ]:
if loaded is not None:
    result = loaded
    print("experiment already exists, loaded.") 
else:
    result = run_experiment(args, M, config, x1, filters, t[:t_final],
                            logger, root, device=device,
                            filters_Q=filters_Q, filters_Phi=filters_Phi,
                            potentials_save_dir=potentials_dir)

# Analyze results 

In [ ]:
theta_t = result['theta_t']
print(theta_t.shape, theta_t.device)
theta_reg_t = result['Theta_reg']

In [ ]:
theta_reg_t[:,5]

## Coarse grain 

In [ ]:
# Move to CPU and convert to numpy for easier plotting
import matplotlib.pyplot as plt
theta_t = theta_t.cpu()
theta_reg_t = theta_reg_t.cpu()

# Get the number of time steps (rows) for each
len_t = theta_t.shape[0]
len_reg = theta_reg_t.shape[0]
print(len_t, len_reg) 

# Iterate through each feature/column
%matplotlib inline
for i in range(theta_t.shape[1]): 
    # Create matching X-axes spanning from 0 to 1 (or 0 to len_t if preferred)
    x_t = np.linspace(0, 1, len_t)
    x_reg = np.linspace(0, 1, len_reg)
    
    plt.figure(figsize=(10, 4))
    
    # Plot both on the same axes
    #plt.plot(results[key]['xt'], theta_t[:, i], label=f"theta_t (pts: {len_t})", alpha=0.8) 
    plt.plot(x_t, theta_t[:, i], label=f"theta_t (pts: {len_t})", alpha=0.8) 
    plt.plot(x_reg, theta_reg_t[:, i], label=f"theta_reg_t (pts: {len_reg})", alpha=0.8) 
    
    plt.title(f"Experiment {label} - Feature {i} Superimposed")
    plt.xlabel("Normalized Time / Progress")
    plt.ylabel("Value")
    plt.legend()
    plt.show()  # Show the combined plot

## Moment matching diagnostics

In [ ]:
threshold = 1e-8
%matplotlib inline
print(label)
if result.get('barphi_e') is None or result.get('barphi_p') is None:
    print('Moment matching arrays not available. Set FORCE_RERUN=True or make sure the auxiliary file exists.')
else:
    plot_moment_matching(result['barphi_e'], result['barphi_p'], result['t'], threshold)
    plt.suptitle(label)
    
    plt.show()


## Visual comparison of samples

In [ ]:
n_groups = x1.shape[0] // 10 # decides how many to plot 

for i in range(n_groups):
    save_config = {
        "filename": str(fig_dir / f"time_series_compare{i}.png"),
        "title": label,
    }
    %matplotlib inline
    Compare_time_series_row(x1[i*5:i*5+5], result['xt'][i*5:i*5+5], 1)

## Time asymmetry 
Check for time asymmetry by investigating the energy increments distribution and skewness vs tau. 

In [ ]:
T = x1.shape[-1]
taus = logspaced_taus(50, num_points=6)              # positive only, for PDFs
pdf_data  = pdf_energy_increments(x1, taus, mode="power")
pdf_synth = pdf_energy_increments(result["xt"], taus, mode="power")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_pdf_panel(pdf_data, ax=axes[0], title="data")
plot_pdf_panel(pdf_synth, ax=axes[1], title="synth")

plot_pdf_folded_grid(pdf_data, pdf_synth, title="right P(x) vs folded-left P(-x)")

fig2, ax2 = plt.subplots()
%matplotlib inline
plot_skewness_comparison(x1, result["xt"], ax=ax2)   # positive-only log-x taus by default

## Leverage with C_pq

In [ ]:
tau, L_data, err_data, L_synth, err_synth = C_pq_plot(x1, result["xt"], p = 3.0, tau_star=10, normalize=False) 

In [ ]:
# 2. Find the midpoint index
half = L_data.shape[0] // 2 

# --- Rel Diff: DATA ---
A_d, B_d = L_data[:half], L_data[-half:][::-1]
sA_d, sB_d = err_data[:half], err_data[-half:][::-1]

rel_diff_data = (-1*A_d + B_d) / ((A_d + B_d) / 2)
# Propagated error for relative difference
err_rel_diff_data = (4 / (A_d + B_d)**2) * np.sqrt((B_d * sA_d)**2 + (A_d * sB_d)**2)

# --- Rel Diff: SYNTHETIC ---
A_s, B_s = L_synth[:half], L_synth[-half:][::-1]
sA_s, sB_s = err_synth[:half], err_synth[-half:][::-1]

rel_diff_synth = (-1*A_s + B_s) / ((A_s + B_s) / 2)
# Propagated error for relative difference
err_rel_diff_synth = (4 / (A_s + B_s)**2) * np.sqrt((B_s * sA_s)**2 + (A_s * sB_s)**2)

# 3. Prepend the midpoint value and its original error (index 0)
diff_data = np.r_[0, rel_diff_data]
diff_synth = np.r_[0, rel_diff_synth]

err_plot_data = np.r_[err_data[half], err_rel_diff_data]
err_plot_synth = np.r_[err_synth[half], err_rel_diff_synth]

# 4. Plotting
plt.figure(figsize=(8, 5))
x_axis = np.arange(len(diff_data))

# Plot lines
plt.plot(x_axis, diff_data, label="Data", color="C0", lw=2)
plt.plot(x_axis, diff_synth, label="Synth", color="C1", lw=2)

# Plot uncertainty bounds
# plt.fill_between(x_axis, diff_data - err_plot_data, diff_data + err_plot_data, 
#                  color="C0", alpha=0.2, label="Data Uncertainty")
# plt.fill_between(x_axis, diff_synth - err_plot_synth, diff_synth + err_plot_synth, 
#                  color="C1", alpha=0.2, label="Synth Uncertainty")

# Reference line at y=0 (dashed, thin)
plt.axhline(0, color="black", linestyle="--", alpha=0.6, label="y = 0")

plt.yscale("log") 
plt.xlabel("Index")
plt.ylabel("Value (Midpoint) / Relative Difference")
plt.title("Symmetry Diagnostic with Propagated Uncertainty")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Scaling exponent ratio 

In [ ]:
taus, ratio_data, ratio_synth = scaling_exponent_ratio_compare(x1, result['xt'], 30, kill_points=1) 

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(taus[:-2], ratio_data[:-2], 'ko-', ms=4, label='Data')
ax.plot(taus[:-2], ratio_synth[:-2], 'ro-', ms=4, label='Synth')
ax.axhline(2.0, color='grey', ls='--', lw=1, label=r'$\zeta_4/\zeta_2=2$ (dimensional)')
ax.set_xscale('log')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'$\zeta_4^L/\zeta_2^L = d\log S_4 / d\log S_2$')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## Wavelet coefficient marginals, structure functions and other diagnostics 

In [ ]:
%matplotlib inline 
def run_project_diagnostics(x_ref, result):
    """
    Uses the existing project plotting utilities. These functions usually create
    their own figures, so we call each diagnostic once per model and add titles.
    """
    diagnostic_functions = [
        ('Spectrum', spec_plot),
        ('Histogram', hist_plot),
        ('Structure functions', structure_plot),
    ]

    for diag_name, diag_fun in diagnostic_functions:
        print('' + '=' * 80)
        print(diag_name)
        print('=' * 80)
        print(label)
        diag_fun(x_ref, result['xt'])
        plt.suptitle(f"{diag_name}: {label}")
        plt.show()

    #print('' + '=' * 80)
    #print('Cross statistics')
    #print('=' * 80)
    #for key, exp in experiments.items():
        #print(exp['label'])
        #cross_plot(x_ref, xt_cg, pq=[(3, 1), (3, 3), (4, 4)])
        #plt.suptitle(f"Cross statistics: {exp['label']}")
        #plt.show()

run_project_diagnostics(x1, result)


## Entropy bound comparison

In [ ]:
def entropy_curves(x_ref, dH_t_bound, t_used):
    d = x_ref.shape[-2] * x_ref.shape[-1]
    H_p_0 = (np.log(2 * np.pi) + 1) * d / 2
    H_t_bound = dH_t_bound.cumsum(0).detach().cpu() / (t_used.shape[0]) + H_p_0
    H_t_gaussian = compute_gaussian_entropy(x_ref, interpolant, t_used) 
    return H_t_bound, H_t_gaussian

plt.figure(figsize=(7, 5))
last_gaussian = None

H_t_bound, H_t_gaussian = entropy_curves(x1, result['dH_t_bound'], result['t'].cpu())

print("Contains NaN:", torch.isnan(result['dH_t_bound']).any())
print("Contains Inf:", torch.isinf(result['dH_t_bound']).any())
last_gaussian = H_t_gaussian
print(label)
print('  Entropy bound:', H_t_bound[-2].item() if torch.is_tensor(H_t_bound[-2]) else H_t_bound[-1])
print('  Gaussian estimation:', H_t_gaussian[-2].item() if torch.is_tensor(H_t_gaussian[-2]) else H_t_gaussian[-2])
plt.plot(result['t'].detach().cpu(), H_t_bound, label=f"bound: {label}")
"""
plt.plot(
    t.detach().cpu(),
    last_gaussian.detach().cpu() if torch.is_tensor(last_gaussian) else last_gaussian,
    '--',
    label='Gaussian entropy estimate',
)
plt.xlabel('t')
plt.ylabel('entropy')
plt.legend()
plt.tight_layout()
plt.show()
"""

## Project entropy plotting utility for each model

## Plots for paper 

In [ ]:
# Usage:
# from paper_figures import make_all_panels
for key, res in result.items(): 
    #data_E = x1
    #synth_E = results[key]["xt"]
    data_lagr = x1
    synth_lagr = result["xt"]
    
    # Plotting panels using specific turbulence variables
    #make_all_panels(data_E, synth_E, data_lagr, synth_lagr, tau_star=20, save_dir=None)          # display
    #make_all_panels(data_E, synth_E, data_lagr, synth_lagr, tau_star=20, save_dir='paper_figs/run1')   # save
    %matplotlib inline
    make_all_panels(data_lagr, synth_lagr, label1="Lagrangian", tau_star=20, save_dir='figures/lagrangian')   # save